# Fine-tuning no Google Colab (GPU T4)

Mesmo pipeline do repositório, com dois parâmetros diferentes: modelo base
maior (`Qwen2.5-1.5B-Instruct`) e `--device cuda`. Em CPU o projeto treina o
`0.5B`; aqui a T4 gratuita permite o `1.5B` em poucos minutos.

**Runtime → Alterar tipo de runtime → T4 GPU** antes de começar.


In [ ]:
!nvidia-smi


## 1. Clonar o projeto e instalar


In [ ]:
!git clone https://github.com/otaviano/fiap-tech-challenge-fase3.git
%cd fiap-tech-challenge-fase3
!pip install -q -e .
!pip install -q -r requirements-finetune.txt


## 2. Conferir o dataset

O dataset processado é versionado no repositório. Para regenerá-lo do zero
(inclusive baixando o MedQuAD), rode as duas linhas comentadas.


In [ ]:
# !./scripts/fetch_medquad.sh
# !med-assist data synth && med-assist data build

import json
resumo = json.load(open('results/eval/resumo_dataset.json'))
print('Treino:', resumo['treino'], '| Validação:', resumo['validacao'])
print(json.dumps(resumo['curadoria']['aprovados_por_categoria'], indent=2, ensure_ascii=False))


## 3. Treinar

LoRA r=16, alpha=32, lr 2e-4, batch efetivo 8. Na T4, três épocas do dataset
completo levam poucos minutos.


In [ ]:
!med-assist finetune train \
    --base-model Qwen/Qwen2.5-1.5B-Instruct \
    --device cuda \
    --epochs 3 \
    --batch-size 2 --grad-accum 4


## 4. Curva de convergência


In [ ]:
import json, matplotlib.pyplot as plt

h = json.load(open('results/eval/finetune_history.json'))
treino = [(r['passo'], r['train_loss']) for r in h['historico'] if 'train_loss' in r]
val = [(r['passo'], r['val_loss']) for r in h['historico'] if 'val_loss' in r]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(*zip(*treino), label='treino', linewidth=1.6)
ax.plot(*zip(*val), 'o--', label='validação', linewidth=1.6)
ax.set_xlabel('passo'); ax.set_ylabel('loss'); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f"LoRA — {h['config']['base_model']}")
plt.tight_layout(); plt.show()

print('Antes: ', h['metricas_iniciais'])
print('Depois:', {k: v for k, v in h['metricas_finais'].items() if 'val' in k})


## 5. Avaliar: base vs. fine-tunado

Mesmos prompts nos dois modelos, critérios objetivos: citação de fonte,
recusa de ato médico, aviso de validação humana e ausência de PII.


In [ ]:
!med-assist finetune evaluate --base-model Qwen/Qwen2.5-1.5B-Instruct --limit 12


## 6. Publicar: merge, GGUF e download

O adapter é fundido nos pesos base e convertido para GGUF quantizado, para
subir no `llama-server` do hospital.


In [ ]:
!med-assist finetune merge --base-model Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /content/llama.cpp/convert_hf_to_gguf.py models/med-assist-merged \
    --outfile models/med-assist-f16.gguf --outtype f16


In [ ]:
# Quantização Q4_K_M exige o binário llama-quantize compilado.
# Alternativa: baixar o f16 e quantizar na máquina que já tem o llama.cpp.
from google.colab import files
files.download('models/med-assist-f16.gguf')


---

Na máquina do hospital:

```bash
llama-quantize med-assist-f16.gguf med-assist-Q4_K_M.gguf Q4_K_M
llama-server -m med-assist-Q4_K_M.gguf --alias med-assist --port 8080
med-assist ask 'Qual a conduta na sepse?' --paciente PRT-2026-100137
```
